# Initialize project environemt

In [ ]:
import sys; from pathlib import Path; sys.path.insert(0, str(Path.home() / "Github" / "the-bible-catalog"))

# Dependencies

In [ ]:
from config.settings import *

In [ ]:
from config._common_libraries import *

In [ ]:
from config._common_functions import *

In [ ]:
from config._util_functions import *

In [ ]:
from config._silver_functions import *

# Parameter(s)

In [ ]:
p_env        = get_env(ENVIRONMENT)
p_debug      = DEBUG
p_job_name   = "transform_esv_bible_embeddings"
p_database   = f"ext_{p_env}"
p_src_schema = "bronze"
p_tgt_schema = "silver"
p_table      = "bible_catalog"
p_embed_model = MODEL['embedding_model']
p_keys       = ["translation", "book", "chapter", "verse_number"]
p_reset      = False

print(f"Job:         {p_job_name}")
print(f"Environment: {p_env}")
print(f"Debug:       {p_debug}")
print(f"Source:      {p_database}.{p_src_schema}.{p_table}")
print(f"Target:      {p_database}.{p_tgt_schema}.{p_table}")
print(f"Model:       {p_embed_model}")
print(f"Keys:        {', '.join(p_keys)}")
print(f"Reset:       {p_reset}")

# Initialize Handler(s)

In [ ]:
timer = ClockHandler()

In [ ]:
messenger = NotificationsHandler(webhook_url=EXTERNAL_SERVICES['discord_webhook'])

In [ ]:
embedder = ESVBibleEmbedding(
    database_name = p_database,
    source_schema = p_src_schema,
    target_schema = p_tgt_schema,
    embed_model   = p_embed_model,
    reset         = p_reset,
)

# Process Embedding(s)

In [ ]:
# Initialize variables
error_message = None
status = "SUCCESS"

# Run process
timer.start()
try:
    # Chunk and embed
    df = embedder.run(key_columns = p_keys,)
except Exception as e:
    status = "FAILED"
    error_message = str(e)
finally:
    timer.end()

# Notify messenger

In [ ]:
messenger.send_workflow_notification(
    workflow_name=f"{p_job_name}-{p_env}",
    job_name=p_job_name,
    status=status,
    timer=timer,
    environment=p_env,
    debug=p_debug,
    custom_message=f"The `{p_job_name}` scheduled notebook procesed successfully. The entire process took approximately {timer.get_elapsed_time()}." if status == "SUCCESS" else f"{p_job_name} failed with error: {error_message}"
)